In [ ]:
"""
Gaussian Splatting for MRI (Notebook‑friendly)
----------------------------------------------
Refactored so that when imported in Jupyter, no CLI args are required.
You can call functions directly without argparse interfering.
"""
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Tuple, Optional

import numpy as np

try:
    import nibabel as nib  # type: ignore
except Exception as e:
    raise ImportError(
        "nibabel is required. Install with `pip install nibabel`.\n"
    ) from e


@dataclass
class GaussianSet:
    mu: np.ndarray
    sigma: float
    w: np.ndarray
    bounds: Optional[Tuple[Tuple[float, float, float], Tuple[float, float, float]]] = None


def _voxel_size_from_affine(affine: np.ndarray) -> Tuple[float, float, float]:
    A = affine[:3, :3]
    sx, sy, sz = np.linalg.norm(A[:, 0]), np.linalg.norm(A[:, 1]), np.linalg.norm(A[:, 2])
    return float(sx), float(sy), float(sz)


def _world_coords_from_indices(affine: np.ndarray, idx: np.ndarray) -> np.ndarray:
    N = idx.shape[0]
    homo = np.ones((N, 4), dtype=np.float32)
    homo[:, :3] = idx.astype(np.float32)
    world = (affine @ homo.T).T[:, :3]
    return world.astype(np.float32)


def load_nifti(path: str) -> Tuple[np.ndarray, np.ndarray]:
    img = nib.load(path)
    data = img.get_fdata(dtype=np.float32)
    affine = img.affine.astype(np.float32)
    finite = np.isfinite(data)
    vmin, vmax = np.min(data[finite]), np.max(data[finite])
    if vmax > vmin:
        data = (data - vmin) / (vmax - vmin)
    else:
        data = np.zeros_like(data, dtype=np.float32)
    return data.astype(np.float32), affine


def extract_gaussians(
    data: np.ndarray,
    affine: np.ndarray,
    intensity_threshold: float = 0.1,
    subsample: int = 4,
    sigma_vox: float = 0.75,
    max_points: int = 500_000,
) -> GaussianSet:
    mask = data >= float(intensity_threshold)
    zz, yy, xx = np.where(mask)
    if subsample > 1:
        sel = (xx % subsample == 0) & (yy % subsample == 0) & (zz % subsample == 0)
        xx, yy, zz = xx[sel], yy[sel], zz[sel]
    N = xx.size
    if N == 0:
        raise ValueError("No voxels passed the threshold.")
    if N > max_points:
        idx = np.random.choice(N, size=max_points, replace=False)
        xx, yy, zz = xx[idx], yy[idx], zz[idx]
    idx_3d = np.stack([xx, yy, zz], axis=1)
    mu = _world_coords_from_indices(affine, idx_3d)
    sx, sy, sz = _voxel_size_from_affine(affine)
    sigma_world = float(sigma_vox * ((sx + sy + sz) / 3.0))
    w = data[zz, yy, xx].astype(np.float32)
    mins, maxs = mu.min(axis=0), mu.max(axis=0)
    return GaussianSet(mu=mu.astype(np.float32), sigma=sigma_world, w=w, bounds=(tuple(mins), tuple(maxs)))


def render_projection(
    gset: GaussianSet,
    axis: str = "z",
    image_size: Tuple[int, int] = (512, 512),
    opacity_scale: float = 1.0,
    clamp: Tuple[float, float] = (0.0, 1.0),
    max_points_render: int = 300_000,
) -> np.ndarray:
    H, W = int(image_size[0]), int(image_size[1])
    mu, w = gset.mu, gset.w
    N = mu.shape[0]
    if N > max_points_render:
        keep = np.random.choice(N, size=max_points_render, replace=False)
        mu, w = mu[keep], w[keep]
    if axis.lower() == "z":
        pts2 = mu[:, [0, 1]]
    elif axis.lower() == "y":
        pts2 = mu[:, [0, 2]]
    elif axis.lower() == "x":
        pts2 = mu[:, [1, 2]]
    else:
        raise ValueError("axis must be one of {'x','y','z'}")
    mins, maxs = np.array(gset.bounds[0][:2]), np.array(gset.bounds[1][:2])
    span = np.maximum(maxs - mins, 1e-6)
    uv = (pts2[:, :2] - mins) / span
    px = np.clip((uv[:, 0] * (W - 1)).round().astype(np.int32), 0, W - 1)
    py = np.clip(((1.0 - uv[:, 1]) * (H - 1)).round().astype(np.int32), 0, H - 1)
    avg_world_span = float((span[0] + span[1]) / 2.0)
    sigma_px = max(0.5, (gset.sigma / max(avg_world_span, 1e-6)) * ((W + H) * 0.5))
    r = int(max(1, math.ceil(3.0 * sigma_px)))
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma_px**2)).astype(np.float32)
    kernel /= (2.0 * math.pi * sigma_px**2)
    canvas = np.zeros((H, W), dtype=np.float32)
    for (ix, iy, wt) in zip(px, py, w):
        if wt <= 0:
            continue
        x0, x1 = max(0, ix - r), min(W, ix + r + 1)
        y0, y1 = max(0, iy - r), min(H, iy + r + 1)
        kx0, kx1 = x0 - (ix - r), kernel.shape[1] - ((ix + r + 1) - x1)
        ky0, ky1 = y0 - (iy - r), kernel.shape[0] - ((iy + r + 1) - y1)
        canvas[y0:y1, x0:x1] += opacity_scale * wt * kernel[ky0:ky1, kx0:kx1]
    lo, hi = clamp
    canvas = np.clip(canvas, lo, hi)
    if hi > 1.0:
        canvas = (canvas - lo) / (hi - lo + 1e-6)
    return canvas.astype(np.float32)

from gaussian_splat_mri import load_nifti, extract_gaussians, render_projection
import matplotlib.pyplot as plt

vol, aff = load_nifti("/home/rbielski/SOOP/ds004889/sub-1/anat/sub-1_T1w.nii.gz")
gset = extract_gaussians(vol, aff, intensity_threshold=0.1, subsample=4)
proj = render_projection(gset, axis='z', image_size=(512,512))

plt.imshow(proj, cmap='gray'); plt.axis('off')


ModuleNotFoundError: No module named 'gaussian_splat_mri'

ModuleNotFoundError: No module named 'gaussian_splat_mri'